In [1]:
import pandas as pd
import re
from ollama import chat, ChatResponse
from tqdm import tqdm  # 用於顯示進度條

In [2]:
# --- 設定 ---
FILE_PATH = '../data/all.xlsx'     # 資料集路徑

In [3]:
# --- 函數：呼叫 Ollama (使用官方庫) ---
def query_ollama(prompt, model='gemma3:12b', context_len=1536, think=False):
    try:
        # 使用 ollama.chat 直接呼叫
        response: ChatResponse = chat(
            model=model,
            messages=[
                {'role': 'user', 'content': prompt}
            ],
            # 您可以在此加入 options，例如 temperature
            options={
                'num_ctx': context_len,  # 在這裡指定 Context Length
                'temperature': 0.0  # 讓輸出結果穩定
            },
            think=think  # 設定為 True 以獲取推理欄位
        )
        # 回傳模型的內容
        return response.message.content
    except Exception as e:
        print(f"Error calling Ollama: {e}")
        return ""

In [4]:
# --- 函數：解析模型輸出 ---
def extract_answer(text):
    """
    從模型的回答中提取選項數字 (1, 2, 3, 4)。
    DeepSeek R1 會輸出 <think>...</think> 思考過程，需要過濾。
    """
    if not text:
        return None

    # 1. 移除 <think> 標籤內的內容 (DeepSeek R1 特有)
    # flags=re.DOTALL 確保 . 可以匹配換行符號
    text_cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    
    # 2. 策略 A: 尋找明確的 "Answer: X" 格式 (Prompt 中有要求)
    match = re.search(r'Answer:\s*([1-4])', text_cleaned, re.IGNORECASE)
    if match:
        return int(match.group(1))
    
    # 3. 策略 B: 如果沒有標準格式，尋找最後出現的數字 1-4
    # 這是一個備用方案
    numbers = re.findall(r'\b([1-4])\b', text_cleaned)
    if numbers:
        return int(numbers[-1])
        
    return None  # 解析失敗

In [5]:
# --- 主程式 ---
# 1. 讀取資料
print(f"正在讀取 {FILE_PATH}...")
try:
    df = pd.read_excel(FILE_PATH)
except Exception as e:
    print(f"讀取檔案失敗: {e}")

# 檢查必要的欄位
required_columns = ['ID', 'Article', 'Question', 'Option1', 'Option2', 'Option3', 'Option4', 'Answer']
if not all(col in df.columns for col in required_columns):
    print(f"錯誤: Excel 檔案缺少必要的欄位。需要: {required_columns}")

正在讀取 ../data/all.xlsx...


In [6]:
df

,ID,Article,Question,Option1,Option2,Option3,Option4,Answer
0,1,明年元旦起，交通部新規定，汽車輪胎胎紋深度將納入定期檢驗項目之一，一旦深度未達1.6公厘，及...,汽車胎紋未到達多少公厘將會被吊銷牌照?,1.5公厘,1.4公厘,1.3公厘,1.6公厘,4
1,2,塑化劑是一種「環境荷爾蒙」，又稱內分泌干擾物，指會干擾生物體內分泌之外因性化學物質。若長時間...,若長時間使用哪一種塑化劑會使男嬰生殖器發育不良、女童性早熟，以及成年男性精蟲減少等問題。,DEHP,DINP,DNOP,DIDP,1
2,3,紐西蘭的海岸邊近日出現數百具「小藍企鵝」的屍體，專家指出，這些企鵝都是被活活餓死的，會有企鵝...,使此處海水變暖的原因為何?,洋流,全球暖化,反聖嬰現象,核能電廠排放廢水,3
3,4,綜合報導，科學期刊《當代生物學》昨天刊登一份美澳聯合研究，研究指出，近年澳洲大堡礁北部的新出...,為甚麼雄性海龜出生率降低，雌性海龜出生率則升高?,機率問題,全球暖化,環境賀爾蒙影響,當地環境影響,2
4,5,花蓮強震發生後，是否還可能發生更大的地震？中央大學地科系教授馬國鳳警告，昨晚花蓮強震應該是主...,這次花蓮大地震主要是哪一地震斷層所影響?,奇美斷層,米崙斷層,玉里斷層,嶺頂斷層,2
...,...,...,...,...,...,...,...,...
8545,8546,臺北市位於木柵地區的福德坑垃圾掩埋場變身「台北能源之丘」，成為全台首座由垃圾場轉型的太陽能發...,下列何者為台北能源之丘計畫?,全國第一個完工發電的垃圾掩埋場轉型風力發電廠,全國第一個完工發電的垃圾掩埋場轉型太陽能電廠,全國第一個完工發電的垃圾掩埋場轉型水力發電廠,全國第一個完工發電的垃圾掩埋場轉型核能電廠,2
8546,8547,臺北市位於木柵地區的福德坑垃圾掩埋場變身「台北能源之丘」，成為全台首座由垃圾場轉型的太陽能發...,垃圾掩埋場轉型的太陽能電廠，預估ㄧ年發電量最高可達幾萬度?,200萬,300萬,2000萬,20000萬,1
8547,8548,春雨不足，加上核一廠機二號機六月初停擺，今天夏天供電情況相當嚴峻，尤其水力供電要看天吃飯，如...,什麼力供電要看天氣?,火力,電力,水力,風力,3
8548,8549,春雨不足，加上核一廠機二號機六月初停擺，今天夏天供電情況相當嚴峻，尤其水力供電要看天吃飯，如...,台電董事長是誰?,朱文章,朱文,朱成,朱文成,4


In [7]:
# --- 主程式 ---
def main():
    correct_count = 0
    total_questions = len(df)
    results = []

    print(f"開始使用 {MODEL_NAME} 處理 {total_questions} 個問題...")

    # 2. 遍歷每一列進行預測
    for index, row in tqdm(df.iterrows(), total=total_questions):
        article = row['Article']
        question = row['Question']
        # 建立選項字典方便取用
        options = {
            1: row['Option1'],
            2: row['Option2'],
            3: row['Option3'],
            4: row['Option4']
        }
        actual_answer = row['Answer']
        #print(actual_answer)

        # 建構 Prompt
        # 這裡的設計是為了讓 DeepSeek R1 能夠先思考再回答，並最後輸出固定格式
        prompt = f"""
Read the following article and answer the multiple-choice question.

Article:
{article}

Question:
{question}

Options:
1. {options[1]}
2. {options[2]}
3. {options[3]}
4. {options[4]}

Instruction:
Analyze the article and select the best option. 
Finally, output the answer in the exact format: "Answer: X", where X is the option number (1, 2, 3, or 4).
Do not output anything else after the answer.
"""
        # 呼叫模型
        response_text = query_ollama(prompt, model=MODEL_NAME, think=THINK_MODE)
        
        # 解析答案
        predicted_answer = extract_answer(response_text)
        
        # 比對結果
        is_correct = False
        try:
            # 處理 Excel 讀入的答案可能是浮點數或字串的情況
            actual_answer_int = int(actual_answer)
            if predicted_answer is not None and predicted_answer == actual_answer_int:
                correct_count += 1
                is_correct = True
            else:
                print("Article:", article)
                print("Question:", question)
                print("Option:", "1.", row['Option1'], "2.", row['Option2'], "3.", row['Option3'], "4.", row['Option4'])
                print("Answer:", actual_answer)
                print("Predicted Answer:", predicted_answer)
                print('-'*30)
        except ValueError:
            pass # 忽略無效的標準答案格式

        # 儲存詳細結果
        results.append({
            "ID": row['ID'],
            "Actual": actual_answer,
            "Predicted": predicted_answer,
            "Correct": is_correct,
            "Model_Response": response_text
        })

        #print(predicted_answer)
        if (index+1) % 100 == 0:
            print(correct_count, " / ", (index+1), " = ", 100.0*correct_count/(index+1), "%")
            break

    # 3. 計算並輸出正確率
    accuracy = (correct_count / total_questions) * 100 if total_questions > 0 else 0
    print("\n" + "="*30)
    print(f"處理完成！")
    print(f"總題數: {total_questions}")
    print(f"答對題數: {correct_count}")
    print(f"正確率 (Accuracy): {accuracy:.2f}%")
    print("="*30)

    # 4. 將結果存檔
    output_file = "result_analysis.xlsx"
    pd.DataFrame(results).to_excel(output_file, index=False)
    print(f"詳細結果已儲存至 {output_file}")

In [8]:
MODEL_NAME = 'gemma3:12b'  # 模型名稱
THINK_MODE = False
main()

開始使用 gemma3:12b 處理 8550 個問題...


  1%|▍                                                                             | 52/8550 [00:54<2:20:40,  1.01it/s]

Article: 在這種極端低溫下，粒子物質波的波長很長、粒子與粒子間的物質波有很大部分重疊，形成一種稱為玻色─愛因斯坦凝聚的物態，也就是玻色子粒子冷卻到接近絕對零度時，呈現出一種氣態、零黏度的超流體狀態，此時我們熟悉的物理規則退到幕後，量子力學開始接管，我們會看到原子行為不再像粒子，反而更像波浪。
Question: 何種情況下會形成玻色─愛因斯坦凝聚的物態?
Option: 1. 宇宙背景輻射溫度 3K 2. 氣體原子雲溫度冷卻至0.0001K 3. 極端低溫下 4. 極端高溫下
Answer: 3
Predicted Answer: 2
------------------------------


  1%|▉                                                                             | 99/8550 [01:36<2:16:59,  1.03it/s]

99  /  100  =  99.0 %

處理完成！
總題數: 8550
答對題數: 99
正確率 (Accuracy): 1.16%
詳細結果已儲存至 result_analysis.xlsx


In [9]:
MODEL_NAME = 'qwen3:8b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 qwen3:8b 處理 8550 個問題...


  0%|▎                                                                               | 30/8550 [00:14<52:53,  2.69it/s]

Article: 來自泰坦星的薩諾斯戴上「無限手套」親自出馬奪取六顆「無限寶石」，其意欲蒐集完「無限寶石」後彈指間隨機抹殺掉宇宙中的一半智慧生命體完成平衡。
Question: 請問《復仇者聯盟3》之中薩諾斯擁有幾顆無限寶石?
Option: 1. 3 2. 4 3. 5 4. 6
Answer: 4
Predicted Answer: 3
------------------------------


  1%|▉                                                                             | 99/8550 [00:42<1:00:33,  2.33it/s]

99  /  100  =  99.0 %

處理完成！
總題數: 8550
答對題數: 99
正確率 (Accuracy): 1.16%
詳細結果已儲存至 result_analysis.xlsx


In [12]:
MODEL_NAME = 'qwen2.5:7b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 qwen2.5:7b 處理 8550 個問題...


  0%|▏                                                                               | 23/8550 [00:20<44:56,  3.16it/s]

Article: 花東縱谷是位於台灣東部的一個縱谷，因橫跨花蓮、台東兩縣而得名，也是上述兩縣的精華地帶所在。有時因地理位置而另稱東台縱谷，在台灣日治時期也被稱為中仙道平野或中仙道。其由菲律賓海板塊與歐亞板塊擠壓形成[3]，夾於中央山脈和海岸山脈之間，南北長約180公里，東西寬2～7公里，面積約1000平方公里，海拔50－250公尺不等。縱谷的地形以沖積平原為主，不少台地夾雜於其中，河流水系分為花蓮溪、秀姑巒溪和卑南溪三大流域，共同塑造出縱谷內的主要地景。
Question: 面向北的方向，花東縱谷的右手邊為哪座山脈?
Option: 1. 玉山 2. 中央山脈 3. 雪山 4. 海岸山脈
Answer: 4
Predicted Answer: 2
------------------------------


  0%|▎                                                                               | 30/8550 [00:23<57:42,  2.46it/s]

Article: 來自泰坦星的薩諾斯戴上「無限手套」親自出馬奪取六顆「無限寶石」，其意欲蒐集完「無限寶石」後彈指間隨機抹殺掉宇宙中的一半智慧生命體完成平衡。
Question: 請問《復仇者聯盟3》之中薩諾斯擁有幾顆無限寶石?
Option: 1. 3 2. 4 3. 5 4. 6
Answer: 4
Predicted Answer: None
------------------------------


...

Article: 腸病毒蠢蠢欲動！2018年新北市已確診4例腸病毒感染併發重症，年齡皆小於5歲。提醒民眾提高警覺務必要養成正確勤洗手的觀念，尤其5歲以下嬰幼兒是腸病毒併發重症的高危險族群，家長要小心防範。

新北市衛生局監測，上週腸病毒門急診就診1,211人次，雖然尚未跨過流行閾值1,900人次的門檻，但近期氣溫明顯上升，預計6月初將進入流行期。依全國社區監測結果，檢出病毒型別以克沙奇A型為多，但社區中仍有多例散發性腸病毒71型個案，顯示腸病毒71型在社區中持續流行。


Question: 今年新北市已確診4例腸病毒感染併發重症，年紀都不到幾歲?
Option: 1. 2 2. 3 3. 4 4. 5
Answer: 4
Predicted Answer: None
------------------------------


  1%|▉                                                                               | 94/8550 [00:46<51:46,  2.72it/s]

Article: 24日上午11點多宜蘭一位林先生開車行經國道5號，沒想到竟看到一對男女騎乘一輛紅牌機車，大喇喇的在國道上緩慢行駛欣賞沿途美景。經警方了解後，發現他們是來台旅遊的香港情侶，因為不熟路況才會使用導航，沒想到就這樣導上國道，最後警方依法開罰3000到6000元罰鍰。
Question: 事件發生於國道幾號?
Option: 1. 1號 2. 3號 3. 5號 4. 7號
Answer: 3
Predicted Answer: None
------------------------------


  1%|▉                                                                             | 99/8550 [00:48<1:08:42,  2.05it/s]

96  /  100  =  96.0 %

處理完成！
總題數: 8550
答對題數: 96
正確率 (Accuracy): 1.12%
詳細結果已儲存至 result_analysis.xlsx


In [14]:
MODEL_NAME = 'qwen3:14b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 qwen3:14b 處理 8550 個問題...


  0%|▏                                                                             | 23/8550 [00:35<3:41:25,  1.56s/it]

Article: 花東縱谷是位於台灣東部的一個縱谷，因橫跨花蓮、台東兩縣而得名，也是上述兩縣的精華地帶所在。有時因地理位置而另稱東台縱谷，在台灣日治時期也被稱為中仙道平野或中仙道。其由菲律賓海板塊與歐亞板塊擠壓形成[3]，夾於中央山脈和海岸山脈之間，南北長約180公里，東西寬2～7公里，面積約1000平方公里，海拔50－250公尺不等。縱谷的地形以沖積平原為主，不少台地夾雜於其中，河流水系分為花蓮溪、秀姑巒溪和卑南溪三大流域，共同塑造出縱谷內的主要地景。
Question: 面向北的方向，花東縱谷的右手邊為哪座山脈?
Option: 1. 玉山 2. 中央山脈 3. 雪山 4. 海岸山脈
Answer: 4
Predicted Answer: 2
------------------------------


  1%|▉                                                                             | 99/8550 [04:24<6:16:22,  2.67s/it]

99  /  100  =  99.0 %

處理完成！
總題數: 8550
答對題數: 99
正確率 (Accuracy): 1.16%
詳細結果已儲存至 result_analysis.xlsx


In [10]:
MODEL_NAME = 'deepseek-r1'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 deepseek-r1 處理 8550 個問題...


  0%|▏                                                                             | 23/8550 [00:12<1:01:26,  2.31it/s]

Article: 花東縱谷是位於台灣東部的一個縱谷，因橫跨花蓮、台東兩縣而得名，也是上述兩縣的精華地帶所在。有時因地理位置而另稱東台縱谷，在台灣日治時期也被稱為中仙道平野或中仙道。其由菲律賓海板塊與歐亞板塊擠壓形成[3]，夾於中央山脈和海岸山脈之間，南北長約180公里，東西寬2～7公里，面積約1000平方公里，海拔50－250公尺不等。縱谷的地形以沖積平原為主，不少台地夾雜於其中，河流水系分為花蓮溪、秀姑巒溪和卑南溪三大流域，共同塑造出縱谷內的主要地景。
Question: 面向北的方向，花東縱谷的右手邊為哪座山脈?
Option: 1. 玉山 2. 中央山脈 3. 雪山 4. 海岸山脈
Answer: 4
Predicted Answer: 2
------------------------------


  1%|▉                                                                               | 99/8550 [00:37<52:50,  2.67it/s]


99  /  100  =  99.0 %

處理完成！
總題數: 8550
答對題數: 99
正確率 (Accuracy): 1.16%
詳細結果已儲存至 result_analysis.xlsx
